# Bedrock Experimentation Notebook

**Purpose**: Experiment with LLM context and prompts using AWS Bedrock

**Features**:
- Easy model switching (Claude / Nova)
- Load context from multiple sources
- Track experiments (model, tokens, cost)

---
## 1. Setup

In [ ]:
import boto3
import json
import pandas as pd
from datetime import datetime
from typing import Optional
from pathlib import Path

# Create experiments directory
Path("experiments").mkdir(exist_ok=True)

---
## 2. Bedrock Client

Unified client that handles Claude and Nova format differences.

In [ ]:
# Available models in Sydney (ap-southeast-2)
MODELS = {
    # Claude (Anthropic)
    "claude-haiku": "anthropic.claude-3-haiku-20240307-v1:0",
    "claude-sonnet": "anthropic.claude-3-sonnet-20240229-v1:0",
    "claude-sonnet-3.5": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    # Nova (Amazon)
    "nova-micro": "amazon.nova-micro-v1:0",
    "nova-lite": "amazon.nova-lite-v1:0",
    "nova-pro": "amazon.nova-pro-v1:0",
}

# Pricing per 1M tokens (USD)
PRICING = {
    "anthropic.claude-3-haiku-20240307-v1:0": {"input": 0.25, "output": 1.25},
    "anthropic.claude-3-sonnet-20240229-v1:0": {"input": 3.00, "output": 15.00},
    "anthropic.claude-3-5-sonnet-20241022-v2:0": {"input": 3.00, "output": 15.00},
    "amazon.nova-micro-v1:0": {"input": 0.035, "output": 0.14},
    "amazon.nova-lite-v1:0": {"input": 0.06, "output": 0.24},
    "amazon.nova-pro-v1:0": {"input": 0.80, "output": 3.20},
}

# Context window limits
CONTEXT_LIMITS = {
    "anthropic.claude-3-haiku-20240307-v1:0": 200_000,
    "anthropic.claude-3-sonnet-20240229-v1:0": 200_000,
    "anthropic.claude-3-5-sonnet-20241022-v2:0": 200_000,
    "amazon.nova-micro-v1:0": 128_000,
    "amazon.nova-lite-v1:0": 300_000,
    "amazon.nova-pro-v1:0": 300_000,
}

In [ ]:
class BedrockClient:
    """Unified client for Bedrock models (Claude and Nova)."""
    
    def __init__(self, model: str = "claude-haiku", region: str = "ap-southeast-2"):
        """
        Initialize client.
        
        Args:
            model: Model alias (e.g., 'claude-haiku', 'nova-lite') or full model ID
        """
        self.client = boto3.client("bedrock-runtime", region_name=region)
        self.model_id = MODELS.get(model, model)  # Allow alias or full ID
        self.model_alias = model if model in MODELS else self._get_alias(self.model_id)
        self.provider = "anthropic" if self.model_id.startswith("anthropic.") else "amazon"
    
    def _get_alias(self, model_id: str) -> str:
        for alias, mid in MODELS.items():
            if mid == model_id:
                return alias
        return model_id
    
    def _build_body(self, prompt: str, system: Optional[str], 
                    max_tokens: int, temperature: float) -> dict:
        if self.provider == "anthropic":
            body = {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": [{"role": "user", "content": prompt}]
            }
            if system:
                body["system"] = system
        else:
            body = {
                "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
                "messages": [{"role": "user", "content": [{"text": prompt}]}]
            }
            if system:
                body["system"] = [{"text": system}]
        return body
    
    def _parse_response(self, result: dict) -> dict:
        if self.provider == "anthropic":
            content = result["content"][0]["text"]
            usage = result.get("usage", {})
            input_tokens = usage.get("input_tokens", 0)
            output_tokens = usage.get("output_tokens", 0)
        else:
            content = result["output"]["message"]["content"][0]["text"]
            usage = result.get("usage", {})
            input_tokens = usage.get("inputTokens", 0)
            output_tokens = usage.get("outputTokens", 0)
        
        return {
            "content": content,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
        }
    
    def invoke(self, prompt: str, system: Optional[str] = None,
               max_tokens: int = 4096, temperature: float = 0.0) -> dict:
        """Send prompt to model and return response with usage stats."""
        response = self.client.invoke_model(
            modelId=self.model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps(self._build_body(prompt, system, max_tokens, temperature))
        )
        result = self._parse_response(json.loads(response["body"].read()))
        result["model"] = self.model_alias
        result["model_id"] = self.model_id
        result["cost"] = self._calculate_cost(result["input_tokens"], result["output_tokens"])
        return result
    
    def _calculate_cost(self, input_tokens: int, output_tokens: int) -> float:
        pricing = PRICING.get(self.model_id, {"input": 0, "output": 0})
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        return input_cost + output_cost
    
    def get_context_limit(self) -> int:
        return CONTEXT_LIMITS.get(self.model_id, 200_000)
    
    def __repr__(self):
        return f"BedrockClient(model='{self.model_alias}', provider='{self.provider}')"

In [ ]:
# Helper function to quickly switch models
def get_client(model: str = "claude-haiku") -> BedrockClient:
    """Get a Bedrock client for the specified model.
    
    Available models:
        Claude: 'claude-haiku', 'claude-sonnet', 'claude-sonnet-3.5'
        Nova:   'nova-micro', 'nova-lite', 'nova-pro'
    """
    return BedrockClient(model=model)

# Quick test
print("Available models:", list(MODELS.keys()))

---
## 3. Test Connection

In [ ]:
# Test with default model (Claude Haiku - cheapest)
client = get_client("claude-haiku")
print(f"Using: {client}")
print(f"Context limit: {client.get_context_limit():,} tokens\n")

response = client.invoke("Say 'Connection successful!' and nothing else.")
print(f"Response: {response['content']}")
print(f"Tokens: {response['input_tokens']} in / {response['output_tokens']} out")
print(f"Cost: ${response['cost']:.6f}")

In [ ]:
# Easy to switch models
client = get_client("nova-lite")
response = client.invoke("What is 2 + 6? Reply with just the number.")
print(f"Response: {response['content']}")

---
## 4. Experiment Tracker

Log each experiment with model, tokens, cost, and notes.

In [ ]:
class ExperimentTracker:
    """Track experiments and save results."""
    
    def __init__(self, name: str):
        self.name = name
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiments = []
    
    def log(self, response: dict, prompt_preview: str = "", notes: str = ""):
        """Log an experiment result."""
        self.experiments.append({
            "timestamp": datetime.now().isoformat(),
            "model": response.get("model", ""),
            "input_tokens": response.get("input_tokens", 0),
            "output_tokens": response.get("output_tokens", 0),
            "total_tokens": response.get("total_tokens", 0),
            "cost_usd": response.get("cost", 0),
            "prompt_preview": prompt_preview[:100] + "..." if len(prompt_preview) > 100 else prompt_preview,
            "response_preview": response.get("content", "")[:200] + "..." if len(response.get("content", "")) > 200 else response.get("content", ""),
            "notes": notes,
        })
        print(f"✓ Logged: {response.get('model')} | {response.get('total_tokens'):,} tokens | ${response.get('cost', 0):.6f}")
    
    def summary(self) -> pd.DataFrame:
        """Return summary as DataFrame."""
        return pd.DataFrame(self.experiments)
    
    def total_cost(self) -> float:
        return sum(e["cost_usd"] for e in self.experiments)
    
    def total_tokens(self) -> int:
        return sum(e["total_tokens"] for e in self.experiments)
    
    def save(self, path: str = None):
        """Save experiments to CSV."""
        if path is None:
            path = f"experiments/{self.name}_{self.timestamp}.csv"
        df = self.summary()
        df.to_csv(path, index=False)
        print(f"\nSaved to: {path}")
        print(f"Total experiments: {len(self.experiments)}")
        print(f"Total tokens: {self.total_tokens():,}")
        print(f"Total cost: ${self.total_cost():.4f}")
        return path

---
## 5. Context Loading Utilities

Load context from various sources.

In [ ]:
def load_text_file(path: str) -> str:
    """Load text from file (.txt, .md, etc.)"""
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def load_excel_as_text(path: str, sheet_name: str = None) -> str:
    """Load Excel file and convert to text format."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    return df.to_string()

def load_csv_as_text(path: str) -> str:
    """Load CSV file and convert to text format."""
    df = pd.read_csv(path)
    return df.to_string()

def estimate_tokens(text: str) -> int:
    """Rough token estimate (~4 chars per token for English)."""
    return len(text) // 4

def context_summary(text: str, name: str = "Context"):
    """Print summary of context size."""
    chars = len(text)
    tokens_est = estimate_tokens(text)
    lines = text.count('\n') + 1
    print(f"{name}:")
    print(f"  Characters: {chars:,}")
    print(f"  Lines: {lines:,}")
    print(f"  Est. tokens: ~{tokens_est:,}")
    return tokens_est

---
## 6. Experiment: Analyze Business Rules with LLM

Load your business rules and let the LLM identify patterns.

### 6.1 Load Context

Choose how to load your business rules:

In [ ]:
# ============================================================
# OPTION A: Paste your rules directly
# ============================================================

BUSINESS_RULES = """
Paste your Confluence content or business rules here.

Example:
- Rule 1: If merchant contains 'WOOLWORTHS', categorise as GROCERIES
- Rule 2: If merchant contains 'UBER' and amount < $50, categorise as TRANSPORT
- etc.
"""

context_summary(BUSINESS_RULES, "Business Rules")

In [ ]:
# ============================================================
# OPTION B: Load from file
# ============================================================

# Uncomment and modify path as needed:

# From text/markdown file:
# BUSINESS_RULES = load_text_file("path/to/rules.md")

# From Excel:
# BUSINESS_RULES = load_excel_as_text("path/to/rules.xlsx", sheet_name="Sheet1")

# From CSV:
# BUSINESS_RULES = load_csv_as_text("path/to/rules.csv")

# context_summary(BUSINESS_RULES, "Business Rules")

In [ ]:
# ============================================================
# OPTION C: Combine multiple sources
# ============================================================

# confluence_rules = load_text_file("confluence_export.md")
# excel_rules = load_excel_as_text("rules_spreadsheet.xlsx")
# 
# COMBINED_CONTEXT = f"""
# === BUSINESS RULES FROM CONFLUENCE ===
# {confluence_rules}
# 
# === RULES FROM SPREADSHEET ===
# {excel_rules}
# """
# 
# context_summary(COMBINED_CONTEXT, "Combined Context")

### 6.2 Start Experiment

In [ ]:
# Initialize tracker for this experiment
tracker = ExperimentTracker(name="rules_analysis")

### 6.3 Ask LLM to Analyze Rules

### 6.3a Read Bank Pattern input file

In [ ]:
import email
import quopri
from bs4 import BeautifulSoup

def load_confluence_doc(path: str) -> str:
    """Parse Confluence's fake .doc export (MIME-encoded HTML)."""
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Parse as MIME message
    msg = email.message_from_string(content)
    
    # Extract HTML part
    html_content = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/html":
                payload = part.get_payload(decode=False)
                # Decode quoted-printable
                html_content = quopri.decodestring(payload.encode()).decode('utf-8', errors='ignore')
                break
    else:
        html_content = msg.get_payload(decode=True).decode('utf-8', errors='ignore')
    
    # Strip HTML tags, keep text
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Extract text with basic structure
    lines = []
    for element in soup.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li', 'td', 'th']):
        tag = element.name
        text = element.get_text(strip=True)
        if not text:
            continue
        if tag == 'h1':
            lines.append(f"\n# {text}")
        elif tag == 'h2':
            lines.append(f"\n## {text}")
        elif tag == 'h3':
            lines.append(f"\n### {text}")
        elif tag == 'h4':
            lines.append(f"\n#### {text}")
        elif tag == 'li':
            lines.append(f"- {text}")
        else:
            lines.append(text)
    
    return "\n".join(lines)

In [ ]:
!ls assets/

In [ ]:
# Load your file
BUSINESS_RULES = load_confluence_doc("assets/bank_patterns.doc")

# Check size
context_summary(BUSINESS_RULES, "Business Rules")

# Preview
print("\n--- Preview ---")
print(BUSINESS_RULES[:2000])

In [ ]:
# Choose model (start cheap)
client = get_client("claude-haiku")
print(f"Using: {client}")
print(f"Context size: ~{estimate_tokens(BUSINESS_RULES):,} tokens")

# Initialize tracker
tracker = ExperimentTracker(name="rules_understanding")

In [ ]:
prompt = f"""
I'm sharing a document containing business rules for categorising financial transactions.

Please:

1. **Summarize your understanding** of these rules in 10-12 bullet points, focusing on the key logic and patterns.

2. **Identify the structure** of the rules:
   - How are they currently organized?
   - Are there clear categories, hierarchies, or groupings?
   - Is the structure consistent throughout?

3. **Suggest improvements** to the structure:
   - What would be a clearer way to organize these rules?
   - Are there redundancies or gaps you noticed?
   - What's missing that would help someone apply these rules?

=== BUSINESS RULES DOCUMENT ===
{BUSINESS_RULES}
=== END DOCUMENT ===

Please be specific and reference actual content from the document where relevant.
"""

print(f"Prompt size: ~{estimate_tokens(prompt):,} tokens")

# Initialize tracker
tracker = ExperimentTracker(name="rules_understanding")

In [ ]:
prompt = f"""
I'm sharing a document containing business rules for categorising financial transactions.

Please:

1. **Summarize your understanding** of these rules in 10-12 bullet points, focusing on the key logic and patterns.

2. **Identify the structure** of the rules:
   - How are they currently organized?
   - Are there clear categories, hierarchies, or groupings?
   - Is the structure consistent throughout?

3. **Suggest improvements** to the structure:
   - What would be a clearer way to organize these rules?
   - Are there redundancies or gaps you noticed?
   - What's missing that would help someone apply these rules?

=== BUSINESS RULES DOCUMENT ===
{BUSINESS_RULES}
=== END DOCUMENT ===

Please be specific and reference actual content from the document where relevant.
"""

# Estimate tokens and cost BEFORE sending
prompt_tokens = estimate_tokens(prompt)
pricing = PRICING[client.model_id]
estimated_input_cost = (prompt_tokens / 1_000_000) * pricing["input"]

print(f"Prompt size: ~{prompt_tokens:,} tokens")
print(f"Estimated input cost: ${estimated_input_cost:.4f}")
print(f"(Output cost depends on response length)")

In [ ]:
# Send to LLM
response = client.invoke(prompt, max_tokens=4096)

# Log experiment
tracker.log(response, prompt_preview="Rules understanding + structure", notes="Initial analysis")

# Display
print("\n" + "="*60)
print("LLM ANALYSIS")
print("="*60)
print(response["content"])


In [ ]:
prompt = f"""
Based on the business rules document below, please reorganize and document ALL the rules in a clear, structured format.

## Requirements

### 1. GENERAL PATTERNS
Start with rules that apply universally (across all providers):
- Separate into **SPEND** vs **NON-SPEND** categories
- For each rule, specify:
  - Pattern/condition (what triggers this rule)
  - base_category_type or category_key
  - Any amount thresholds or special conditions

### 2. PROVIDER-SPECIFIC PATTERNS
Then document rules that only apply to specific providers (e.g., ANZ, Macquarie, etc.):
- Group by provider
- For each rule, specify:
  - Provider name
  - Pattern/condition
  - base_category_type or category_key
  - Why this differs from the general pattern (if known)

### 3. FORMAT
Please use **tables** for clarity. Example format:

**General Spend Patterns:**
| Pattern/Condition | Category Key | Notes |
|-------------------|--------------|-------|
| Merchant contains "WOOLWORTHS" | GROCERIES | ... |

**Provider-Specific Patterns (ANZ):**
| Pattern/Condition | Category Key | Differs From General | Notes |
|-------------------|--------------|---------------------|-------|
| ... | ... | ... | ... |

### 4. INCONSISTENCIES & GAPS
At the end, add a section that flags:
- **Inconsistencies**: Rules that conflict or contradict each other
- **Gaps**: Scenarios not covered by any rule
- **Ambiguities**: Rules that are unclear or could be interpreted multiple ways

## Important
- Be COMPREHENSIVE - include ALL rules from the document
- If a rule is unclear, include it anyway and flag it
- Preserve the original category_key/base_category_type names exactly as written

=== BUSINESS RULES DOCUMENT ===
{BUSINESS_RULES}
=== END DOCUMENT ===
"""

# Estimate tokens and cost
prompt_tokens = estimate_tokens(prompt)
pricing = PRICING[client.model_id]
estimated_input_cost = (prompt_tokens / 1_000_000) * pricing["input"]

print(f"Using: {client}")
print(f"Prompt size: ~{prompt_tokens:,} tokens")
print(f"Estimated input cost: ${estimated_input_cost:.4f}")
print(f"\nNote: Requesting comprehensive output - may use significant output tokens")

In [ ]:
# Send to LLM (increase max_tokens for comprehensive output)
response = client.invoke(prompt, max_tokens=8192)

# Log experiment
tracker.log(response, prompt_preview="Reorganize rules - tabular format", notes="Comprehensive restructure")

# Display
print("\n" + "="*60)
print("RESTRUCTURED BUSINESS RULES")
print("="*60)
print(response["content"])

In [ ]:
# Save output to file for reference
output_path = f"experiments/restructured_rules_{tracker.timestamp}.md"
with open(output_path, "w") as f:
    f.write(response["content"])
print(f"\nSaved to: {output_path}")

In [ ]:
# Your current code
#response = client.invoke(prompt, max_tokens=8192)

# To see precise token counts, access the usage metadata:
print(f"Input tokens: {response['usage']['input_tokens']}")
#print(f"Output tokens: {response['usage']['output_tokens']}")
print(f"Total tokens: {response['usage']['input_tokens'] + response['usage']['output_tokens']}")

# Cost breakdown
input_cost = response['usage']['input_tokens'] * 0.00000025  # Haiku input pricing
output_cost = response['usage']['output_tokens'] * 0.00000125  # Haiku output pricing
print(f"Input cost: ${input_cost:.6f}")
print(f"Output cost: ${output_cost:.6f}")
print(f"Total cost: ${input_cost + output_cost:.6f}")

In [ ]:
# ============================================================
# EXPERIMENT: Ask LLM to summarize/simplify rules
# ============================================================

prompt = f"""
Below are business rules for categorising financial transactions.
Please analyze these rules and:

1. Identify the main CATEGORIES used
2. Summarize the GENERAL PATTERNS for categorisation
3. List any PROVIDER-SPECIFIC rules (rules that apply to specific merchants/providers)
4. Note any AMBIGUOUS or CONFLICTING rules

=== BUSINESS RULES ===
{BUSINESS_RULES}
=== END RULES ===

Provide a structured summary.
"""

response = client.invoke(prompt, max_tokens=4096)

# Log experiment
tracker.log(response, prompt_preview=prompt[:100], notes="Initial rules analysis")

# Display response
print("\n" + "="*60)
print("LLM ANALYSIS")
print("="*60)
print(response["content"])

In [ ]:
# ============================================================
# EXPERIMENT: Ask follow-up questions
# ============================================================

# Modify this prompt for your specific questions
follow_up_prompt = f"""
Based on these business rules:

=== BUSINESS RULES ===
{BUSINESS_RULES}
=== END RULES ===

Question: [Your specific question here]

For example:
- How should UBER EATS transactions be categorised?
- What's the rule for recurring payments?
- List all rules related to [specific provider]
"""

# response = client.invoke(follow_up_prompt, max_tokens=2048)
# tracker.log(response, prompt_preview=follow_up_prompt[:100], notes="Follow-up question")
# print(response["content"])

### 6.4 Compare Models

In [ ]:
# ============================================================
# Compare same prompt across different models
# ============================================================

test_prompt = f"""
Summarize these business rules in 5 bullet points:

{BUSINESS_RULES}
"""

models_to_test = ["claude-haiku", "nova-lite"]  # Add more as needed

for model_name in models_to_test:
    client = get_client(model_name)
    response = client.invoke(test_prompt, max_tokens=1024)
    tracker.log(response, prompt_preview="Summarize rules", notes=f"Model comparison: {model_name}")
    print(f"\n--- {model_name} ---")
    print(response["content"][:500] + "..." if len(response["content"]) > 500 else response["content"])

---
## 7. Review & Save Experiments

In [ ]:
# View all experiments in this session
tracker.summary()

In [ ]:
# Save experiments to CSV
tracker.save()

---
## 8. Quick Reference

### Models
```python
client = get_client("claude-haiku")      # Cheapest Claude
client = get_client("claude-sonnet-3.5") # Best Claude
client = get_client("nova-lite")         # Cheap Nova
client = get_client("nova-pro")          # Best Nova
```

### Context Limits
| Model | Context Limit |
|-------|---------------|
| Claude (all) | 200K tokens |
| Nova Micro | 128K tokens |
| Nova Lite/Pro | 300K tokens |

### Cost (per 1M tokens)
| Model | Input | Output |
|-------|-------|--------|
| Claude Haiku | $0.25 | $1.25 |
| Claude Sonnet 3.5 | $3.00 | $15.00 |
| Nova Micro | $0.035 | $0.14 |
| Nova Lite | $0.06 | $0.24 |
| Nova Pro | $0.80 | $3.20 |